# Lab - Supervisor and Subagents: Scope, Facts, Cost

Three tasks that split ShopAssist into a coordinator and two subagents - the way a developer has to run them: each agent confined to its own tools, facts handed over as data, and the cost of every agent counted.

| Task | What you learn |
| --- | --- |
| 1 | One function runs any agent - and refuses tools outside that agent's list |
| 2 | Subagents get the facts they need as a small JSON object, nothing more |
| 3 | The coordinator delegates through agent tools; the run's bill, per agent |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

# One lamp order with two identical charges.
ORDERS = {"ORD-87654321": {"item": "lamp", "status": "delivered",
                           "delivered_days_ago": 45, "price": 60}}
CHARGES = {"ORD-87654321": [{"amount": 60, "date": "day 1"},
                            {"amount": 60, "date": "day 1"}]}


def lookup_order(order_id):
    return ORDERS.get(order_id, {"error": "order not found"})


def list_charges(order_id):
    return CHARGES.get(order_id, [])


def refund_charge(order_id, amount):
    return {"refunded": amount, "order_id": order_id}


def tool(name, description, fields):
    return {"name": name, "description": description,
            "input_schema": {"type": "object", "properties": fields,
                             "required": list(fields)}}


ORDER_TOOLS = [tool("lookup_order", "Look up an order by its ID.",
                    {"order_id": {"type": "string"}})]
BILLING_TOOLS = [tool("list_charges", "List the charges of an order.",
                      {"order_id": {"type": "string"}}),
                 tool("refund_charge", "Refund one charge of an order.",
                      {"order_id": {"type": "string"}, "amount": {"type": "number"}})]
FUNCTIONS = {"lookup_order": lookup_order, "list_charges": list_charges,
             "refund_charge": refund_charge}

RECEIVED = []   # every agent's task and tools, exactly as it received them
USAGE = {}      # agent name -> requests and tokens

---

## Task 1 - Each agent runs only its own tools

In [ ]:
# ============================================================
# TASK 1 - Each agent runs only its own tools
# ============================================================
#
# WHAT TO DO
#   Finish call_tool(): it runs a tool_use block only when the
#   tool is in this agent's own list, and refuses anything else
#   with an error result. run_agent() - one function for every
#   agent - uses it, and counts each agent's requests and tokens.
#
# WHY IT MATTERS
#   Every agent is the same function with different arguments:
#   its own system prompt, its own tools, its own task. Small
#   agents with a narrow job stay sharp - but "narrow" must
#   hold where the tool runs, not only in the list Claude is
#   shown. The order agent must not be able to refund anything,
#   whatever it asks for.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now the key piece: one function that runs any agent".
#
# HOW TO DO IT
#   Three blanks:
#
#       tools          this agent's own tool list
#       True           a refused call is an error result
#       task           the agent's first and only message
# ============================================================

# BEGIN SOLUTION
def call_tool(tools, block):
    allowed = {t["name"] for t in tools}
    if block.name not in allowed:
        output, failed = {"error": f"{block.name} is not allowed for this agent"}, True
    else:
        output, failed = FUNCTIONS[block.name](**block.input), False
    return {"type": "tool_result", "tool_use_id": block.id,
            "content": json.dumps(output), "is_error": failed}


def run_agent(name, system, tools, task):
    RECEIVED.append({"agent": name, "task": task, "tools": [t["name"] for t in tools]})
    bill = USAGE.setdefault(name, {"calls": 0, "input": 0, "output": 0})
    messages = [{"role": "user", "content": task}]
    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=600, output_config={"effort": "low"},
            system=system, tools=tools, messages=messages)
        bill["calls"] += 1
        bill["input"] += response.usage.input_tokens
        bill["output"] += response.usage.output_tokens
        if response.stop_reason != "tool_use":
            return next(b.text for b in response.content if b.type == "text")
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": [
            call_tool(tools, b) for b in response.content if b.type == "tool_use"]})
# SCAFFOLD: def call_tool(tools, block):
# SCAFFOLD:     allowed = {t["name"] for t in ...}   # tools - this agent's own list
# SCAFFOLD:     if block.name not in allowed:
# SCAFFOLD:         output, failed = {"error": f"{block.name} is not allowed for this agent"}, ...   # True
# SCAFFOLD:     else:
# SCAFFOLD:         output, failed = FUNCTIONS[block.name](**block.input), False
# SCAFFOLD:     return {"type": "tool_result", "tool_use_id": block.id,
# SCAFFOLD:             "content": json.dumps(output), "is_error": failed}
# SCAFFOLD:
# SCAFFOLD:
# SCAFFOLD: def run_agent(name, system, tools, task):
# SCAFFOLD:     RECEIVED.append({"agent": name, "task": task, "tools": [t["name"] for t in tools]})
# SCAFFOLD:     bill = USAGE.setdefault(name, {"calls": 0, "input": 0, "output": 0})
# SCAFFOLD:     messages = [{"role": "user", "content": ...}]   # task
# SCAFFOLD:     while True:
# SCAFFOLD:         response = client.messages.create(
# SCAFFOLD:             model=MODEL, max_tokens=600, output_config={"effort": "low"},
# SCAFFOLD:             system=system, tools=tools, messages=messages)
# SCAFFOLD:         bill["calls"] += 1
# SCAFFOLD:         bill["input"] += response.usage.input_tokens
# SCAFFOLD:         bill["output"] += response.usage.output_tokens
# SCAFFOLD:         if response.stop_reason != "tool_use":
# SCAFFOLD:             return next(b.text for b in response.content if b.type == "text")
# SCAFFOLD:         messages.append({"role": "assistant", "content": response.content})
# SCAFFOLD:         messages.append({"role": "user", "content": [
# SCAFFOLD:             call_tool(tools, b) for b in response.content if b.type == "tool_use"]})
# END SOLUTION: replace each ... below with the value named beside it

RECEIVED.clear(); USAGE.clear()
try:
    report = run_agent("order", "You check orders. Report the item, status and delivery "
                       "age in one sentence.", ORDER_TOOLS, "Check order ORD-87654321.")
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print("report:", report)

check("scoped_dispatch", call_tool=call_tool, order_tools=ORDER_TOOLS, report=report)

---

## Task 2 - Pass the facts as data

In [ ]:
# ============================================================
# TASK 2 - Pass the facts as data
# ============================================================
#
# WHAT TO DO
#   Write the two subagents. The billing agent receives the
#   facts the coordinator already has - as a small JSON object
#   inside its task: the order ID and the order agent's report.
#   Run both and look at what each received and what it cost.
#
# WHY IT MATTERS
#   A subagent starts with an empty conversation and sees only
#   its task. What the coordinator forgets to pass, the
#   subagent never knows - and what it passes needlessly, the
#   subagent pays for on every request. Facts as named data
#   keep the hand-over explicit, small and easy to log.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The billing agent gets the billing tools".
#
# HOW TO DO IT
#   Three blanks:
#
#       BILLING_TOOLS   the billing agent's tools
#       order_id        the order, by ID
#       facts           the order agent's report
# ============================================================

try:
    # BEGIN SOLUTION
    def order_agent(order_id):
        return run_agent("order", "You check orders. Report the item, status and delivery "
                         "age in one sentence.", ORDER_TOOLS, f"Check order {order_id}.")

    def billing_agent(order_id, facts):
        handover = json.dumps({"order_id": order_id, "order_report": facts})
        return run_agent("billing", "You fix billing problems. Refund only a duplicate "
                         "charge. Report what you did in one sentence.", BILLING_TOOLS,
                         f"Order facts: {handover}\nCheck the charges and fix any duplicate.")
    # SCAFFOLD: def order_agent(order_id):
    # SCAFFOLD:     return run_agent("order", "You check orders. Report the item, status and delivery "
    # SCAFFOLD:                      "age in one sentence.", ORDER_TOOLS, f"Check order {order_id}.")
    # SCAFFOLD:
    # SCAFFOLD: def billing_agent(order_id, facts):
    # SCAFFOLD:     handover = json.dumps({"order_id": ..., "order_report": ...})   # order_id, facts
    # SCAFFOLD:     return run_agent("billing", "You fix billing problems. Refund only a duplicate "
    # SCAFFOLD:                      "charge. Report what you did in one sentence.", ...,   # BILLING_TOOLS
    # SCAFFOLD:                      f"Order facts: {handover}\nCheck the charges and fix any duplicate.")
    # END SOLUTION: replace each ... below with the value named beside it

    RECEIVED.clear(); USAGE.clear()
    facts = order_agent("ORD-87654321")
    print("billing report:", billing_agent("ORD-87654321", facts))
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for entry in RECEIVED:
    print(entry["agent"], "got", entry["tools"], "and:", repr(entry["task"]))
print("usage:", USAGE)

check("explicit_facts", received=RECEIVED, usage=USAGE)

---

## Task 3 - Delegate, then add up the bill

In [ ]:
# ============================================================
# TASK 3 - Delegate, then add up the bill
# ============================================================
#
# WHAT TO DO
#   Give the coordinator the two subagents as its only tools,
#   map those tool names to the agent functions, run the
#   customer's message - then add up what the whole run cost.
#
# WHY IT MATTERS
#   The coordinator owns the customer and has no data tools;
#   delegating is a tool call. Every hand-over is another agent
#   with its own requests, so a coordinated run costs more than
#   one agent would. The per-agent usage tells you where the
#   tokens went - and whether a subagent is worth its price.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The coordinator sees the two subagents as tools".
#
# HOW TO DO IT
#   Three blanks:
#
#       billing_agent       the function behind the billing_agent tool
#       COORDINATOR_TOOLS   the coordinator's only tools
#       USAGE.values()      every agent's bill
# ============================================================

COORDINATOR_TOOLS = [
    tool("order_agent", "Delegate: check an order.", {"order_id": {"type": "string"}}),
    tool("billing_agent", "Delegate: check and fix the charges of an order. "
         "Pass the order facts you already have.",
         {"order_id": {"type": "string"}, "facts": {"type": "string"}}),
]

RECEIVED.clear(); USAGE.clear()
try:
    # BEGIN SOLUTION
    FUNCTIONS.update({"order_agent": order_agent, "billing_agent": billing_agent})
    answer = run_agent(
        "coordinator",
        "You are ShopAssist's coordinator. You have no data tools. Delegate to the order "
        "agent and the billing agent, pass each one what it needs, and answer the "
        "customer in two sentences.",
        COORDINATOR_TOOLS,
        "I was charged twice for order ORD-87654321. Please fix it.")
    total = sum(u["input"] + u["output"] for u in USAGE.values())
    # SCAFFOLD: FUNCTIONS.update({"order_agent": order_agent, "billing_agent": ...})   # billing_agent
    # SCAFFOLD: answer = run_agent(
    # SCAFFOLD:     "coordinator",
    # SCAFFOLD:     "You are ShopAssist's coordinator. You have no data tools. Delegate to the order "
    # SCAFFOLD:     "agent and the billing agent, pass each one what it needs, and answer the "
    # SCAFFOLD:     "customer in two sentences.",
    # SCAFFOLD:     ...,   # COORDINATOR_TOOLS
    # SCAFFOLD:     "I was charged twice for order ORD-87654321. Please fix it.")
    # SCAFFOLD: total = sum(u["input"] + u["output"] for u in ...)   # USAGE.values()
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print("ShopAssist:", answer)
for name, bill in USAGE.items():
    print(f"  {name:12} {bill['calls']} requests, {bill['input']} in / {bill['output']} out")
print("  total tokens:", total)

check("usage_report", answer=answer, received=RECEIVED, usage=USAGE, total=total)

---

## Done

Give each subagent a narrow job and only the tools for it - enforced where the tool runs - pass the context it needs explicitly, as data, and let the coordinator own the customer. Count every agent's tokens: delegation is not free.